<a href="https://colab.research.google.com/github/svetazo060510/goit-numpr-hw-11/blob/main/HW11_StremedlovskaS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання Тема 11

In [2]:
!pip install pygad

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 3.3 MB/s eta 0:00:00


In [11]:
import time
import pygad
import numpy as np

In [12]:
# 1. Ініціалізація даних
items = np.array(["Ref_A", "Phone", "TV_55", "TV_50", "TV_42", "Note_A", "Ventilator",
                  "Micro_A", "Micro_B", "Micro_C", "Ref_B", "Ref_C", "Note_B", "Note_C"])

spaces = np.array([0.751, 0.0000899, 0.400, 0.290, 0.200, 0.00350, 0.496,
                   0.0424, 0.0544, 0.0319, 0.635, 0.870, 0.498, 0.527])

prices = np.array([999.90, 2199.12, 4346.99, 3999.90, 2999.00, 2499.90, 199.90,
                   308.66, 429.90, 299.29, 849.00, 1199.89, 1999.90, 3999.00])

MAX_SPACE = 3.0
NUM_GENES = len(items)

In [13]:
# 2. Оптимізована фітнес-функція (Метод м'якого штрафу)
def fitness_function(ga_instance, solution, solution_idx):
    total_space = np.sum(solution * spaces)
    total_price = np.sum(solution * prices)

    if total_space <= MAX_SPACE:
        return total_price
    else:
        # Штраф пропорційний ступеню порушення обмеження
        penalty = (total_space - MAX_SPACE) * 10000
        return total_price - penalty

In [21]:
# 3. Функція для запуску експериментів
def run_experiment(name, fitness_fn=fitness_function, crossover="single_point", mutation="random", selection="sss"):
    print(f"RUNNING: {name} (Cross: {crossover}, Mut: {mutation}, Sel: {selection})")

    sol_per_pop = 20
    # Фіксуємо початкову популяцію
    initial_pop = np.random.RandomState(42).randint(0, 2, size=(sol_per_pop, NUM_GENES))

    ga_params = {
        "num_generations": 100,
        "num_parents_mating": 10,
        "fitness_func": fitness_fn,
        "initial_population": initial_pop,
        "gene_type": int,
        "gene_space": [0, 1],
        "parent_selection_type": selection,
        "keep_parents": 2,
        "crossover_type": crossover,
        "mutation_type": mutation,
        "mutation_probability": 0.1,
        "random_seed": 42,
        "suppress_warnings": True
    }

    start_time = time.time()
    ga_instance = pygad.GA(**ga_params)
    ga_instance.run()
    end_time = time.time()

    solution, solution_fitness, solution_idx = ga_instance.best_solution()
    total_space = np.sum(solution * spaces)

    print(f"Time Taken:   {(end_time - start_time):.5f} seconds")
    print(f"Best Fitness: ${solution_fitness:.2f}")
    print(f"Used Space:   {total_space:.3f} m^3 (Limit: {MAX_SPACE})")
    print(f"Chromosome:   {solution.tolist()}")
    print("=" * 60)

In [19]:
# 4. Виконання
run_experiment("Good Baseline", crossover="single_point", mutation="random")
run_experiment("Alternative Config", crossover="two_points", mutation="swap")
run_experiment("Robust Explorer", crossover="uniform", mutation="random", selection="tournament")

RUNNING: Good Baseline (Cross: single_point, Mut: random, Sel: sss)
Time Taken:   0.13710 seconds
Best Fitness: $24281.55
Used Space:   2.917 m^3 (Limit: 3.0)
Chromosome:   [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1]
RUNNING: Alternative Config (Cross: two_points, Mut: swap, Sel: sss)
Time Taken:   0.10539 seconds
Best Fitness: $20482.45
Used Space:   2.886 m^3 (Limit: 3.0)
Chromosome:   [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0]
RUNNING: Robust Explorer (Cross: uniform, Mut: random, Sel: tournament)
Time Taken:   0.18289 seconds
Best Fitness: $23542.99
Used Space:   2.820 m^3 (Limit: 3.0)
Chromosome:   [0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1]


## Висновок

За результатами трьох експериментів встановлено:
* для задачі оптимізації вантажу з 14 товарів найбільш ефективною є базова конфігурація (**Single Point Crossover + SSS Selection**). Вона дозволила отримати максимальний прибуток у $24,281.55.
* Експеримент із Uniform Crossover (**Robust Explorer**) показав високу стабільність, проте не зміг досягти максимуму через надмірну стохастичність (випадковість) на короткій хромосомі.
* Найшвидшим виявився сценарій **Alternative Config**, проте його точність була найнижчою, що свідчить про передчасну збіжність алгоритму.

## Експеримент

Генетичні алгоритми за своєю природою є стохастичними (випадковими) методами пошуку. Вони не гарантують знаходження глобального максимуму, а лише пропонують "достатньо добре" рішення за короткий час. Тому ми вирішили провести перевірку методом Brute Force (повного перебору), щоб:
1. Перевірити точність: Оскільки розмірність нашої задачі невелика ($n = 14$ товарів), кількість усіх можливих комбінацій становить лише $2^{14} = 16\,384$. Це дозволяє знайти абсолютно найкраще рішення математично точним шляхом.
2. Порівняти результат "Good Baseline" із абсолютним ідеалом. Так ми зможемо точно сказати, чи знайшов генетичний алгоритм глобальний максимум, чи застряг у локальному.
3. Порівняти час виконання ідеального пошуку та генетичного алгоритму.

In [20]:
import itertools

def find_absolute_optimum(spaces, prices, max_space):
    start_time = time.time()

    best_price = 0
    best_combination = None
    best_space = 0

    # Кількість комбінацій: 2^14 = 16,384
    for combo in itertools.product([0, 1], repeat=len(spaces)):
        current_combo = np.array(combo)
        total_space = np.sum(current_combo * spaces)

        if total_space <= max_space:
            total_price = np.sum(current_combo * prices)
            if total_price > best_price:
                best_price = total_price
                best_combination = current_combo
                best_space = total_space

    end_time = time.time()
    return best_price, best_space, best_combination, (end_time - start_time)

# Запуск
true_price, true_space, true_chromosome, duration = find_absolute_optimum(spaces, prices, MAX_SPACE)

print("=== ABSOLUTE OPTIMUM (BRUTE FORCE) ===")
print(f"Time Taken:      {duration:.5f} seconds")
print(f"Best Price:      ${true_price:.2f}")
print(f"Space Used:      {true_space:.3f} m^3")
print(f"Chromosome:      {true_chromosome.tolist()}")
print("=" * 60)

=== ABSOLUTE OPTIMUM (BRUTE FORCE) ===
Time Taken:      0.19777 seconds
Best Price:      $24281.55
Space Used:      2.917 m^3
Chromosome:      [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1]


## Фінальний висновок

Порівняльний аналіз результатів генетичного алгоритму та методу повного перебору (Brute Force) підтвердив, що модель Good Baseline генетичного алгоритму знайшла глобальний максимум цільової функції. Генетичний алгоритм також показав вищу швидкість виконання, проаналізувавши лише 1/8 частину простору пошуку. В нашому випадку для 14 товарів повний перебір є обчислювально доступним, але він демонструє експоненціальну складність $O(2^n)$ і якщо ми додамо ще лише 10 товарів ($2^{24}$), Brute Force сповільниться в 1024 рази.